# 🍽️ Restaurant Data Cleanup & Inspection
### LA Luxury Restaurant Recommendation System — Phase 1

**Purpose:** Load, inspect, clean, and validate the `cleaned_restaurants_list.csv` dataset  
**Dataset:** 100 upscale & Michelin-rated restaurants in Los Angeles County  
**Output:** A clean, validated `cleaned_restaurants_final.csv` ready for vector embedding and LLM processing

---
**Columns in Dataset:**
| Column | Description |
|--------|-------------|
| `Name` | Restaurant name |
| `Location` | City/neighborhood |
| `Description` | One-sentence summary |
| `Address` | Full street address |
| `Telephone Number` | Phone number |
| `Price` | Dollar sign scale (\$ to \$\$\$\$\$) |
| `Cuisine Type` | Cuisine category |
| `Dining Atmosphere` | Vibe/atmosphere |
| `Sky-High Rooftop` | Yes/No — rooftop/top-floor location |
| `Michelin-Guide` | Michelin designation or No |
| `Customer Ratings` | Aggregated rating (1.0–5.0) |
| `Operation Hours` | Hours of operation |
| `Reservations` | Reservation policy |
| `Dress Code` | Dress code requirement |

## 📦 Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Display settings — show all columns and full text
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)

print("✅ Libraries loaded successfully.")

## 📂 Cell 2 — Load the Dataset

> **Note:** Place `cleaned_restaurants_list.csv` in the same folder as this notebook,  
> or update the path below to point to your file location.

In [ ]:
# ---------------------------------------------------------
# UPDATE THIS PATH if your CSV is in a different location
# ---------------------------------------------------------
CSV_PATH = "../data/cleaned_restaurants_list.csv"

df = pd.read_csv(CSV_PATH)

print(f"✅ Dataset loaded successfully!")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:>2}. {col}")

## 🔍 Cell 3 — Initial Data Preview

In [ ]:
# Preview the first 10 rows
print("=== FIRST 10 ROWS ===")
df.head(10)

In [ ]:
# Preview the last 10 rows
print("=== LAST 10 ROWS ===")
df.tail(10)

## 📊 Cell 4 — Data Types & Structure

In [ ]:
print("=== DATA TYPES ===")
print(df.dtypes)

print("\n=== DATAFRAME INFO ===")
df.info()

## ❓ Cell 5 — Missing Value Check

In [ ]:
print("=== MISSING VALUES PER COLUMN ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_report)

if missing.sum() == 0:
    print("\n✅ No missing values found in any column.")
else:
    print(f"\n⚠️  Total missing cells: {missing.sum()}")

## 🔢 Cell 6 — Descriptive Statistics (Numeric Columns)

In [ ]:
print("=== DESCRIPTIVE STATISTICS — Customer Ratings ===")
print(df['Customer Ratings'].describe())

print("\n=== RATING DISTRIBUTION ===")
print(df['Customer Ratings'].value_counts().sort_index(ascending=False))

## 📋 Cell 7 — Categorical Column Value Counts

In [ ]:
categorical_cols = [
    'Price', 'Cuisine Type', 'Dining Atmosphere',
    'Sky-High Rooftop', 'Michelin-Guide', 'Reservations', 'Dress Code'
]

for col in categorical_cols:
    print(f"\n=== {col.upper()} ===")
    print(df[col].value_counts().to_string())

## 🔎 Cell 8 — Unique Neighborhoods / Locations

In [ ]:
print("=== UNIQUE LOCATIONS (Neighborhoods) ===")
locations = sorted(df['Location'].unique())
for loc in locations:
    count = (df['Location'] == loc).sum()
    print(f"  {loc:<30} ({count} restaurant{'s' if count > 1 else ''})")

print(f"\nTotal unique neighborhoods: {len(locations)}")

## 🧹 Cell 9 — Data Quality: Trailing Whitespace Check

Whitespace at the start or end of string values can cause silent bugs  
in filtering, searching, and vector embedding. We strip all text columns.

In [ ]:
string_cols = df.select_dtypes(include='object').columns.tolist()

print("=== CHECKING FOR TRAILING/LEADING WHITESPACE ===")
issues_found = False

for col in string_cols:
    mask = df[col].str.strip() != df[col]
    if mask.any():
        issues_found = True
        print(f"\n⚠️  Column '{col}' — {mask.sum()} row(s) with whitespace:")
        for idx in df[mask].index:
            print(f"   Row {idx} ({df.loc[idx, 'Name']}): {repr(df.loc[idx, col][-30:])}")

if not issues_found:
    print("✅ No whitespace issues found.")

# --- FIX: Strip all string columns ---
for col in string_cols:
    df[col] = df[col].str.strip()

print("\n✅ All string columns stripped of leading/trailing whitespace.")

## 🔁 Cell 10 — Duplicate Name Check

Some restaurants intentionally appear twice because they have multiple locations  
(e.g., Pine & Crane in DTLA and Silver Lake, Badmaash in Hollywood and DTLA).  
These are **valid duplicates** — we confirm they have different addresses.

In [ ]:
print("=== DUPLICATE RESTAURANT NAMES ===")
dupes = df[df.duplicated('Name', keep=False)].copy()

if dupes.empty:
    print("✅ No duplicate names found.")
else:
    print(f"Found {len(dupes)} rows with shared names (may be intentional multi-location restaurants):\n")
    print(dupes[['Name', 'Location', 'Address', 'Michelin-Guide']].to_string(index=True))

# --- Check for EXACT duplicate rows (same name AND same address) ---
print("\n=== EXACT DUPLICATE ROWS (same Name + Address) ===")
exact_dupes = df[df.duplicated(subset=['Name', 'Address'], keep=False)]
if exact_dupes.empty:
    print("✅ No exact duplicate rows found — multi-location entries are legitimately distinct.")
else:
    print(f"⚠️  {len(exact_dupes)} rows are exact duplicates:")
    print(exact_dupes[['Name', 'Location', 'Address']].to_string())

## 📍 Cell 11 — Address Format Inspection

Checks for formatting inconsistencies in the `Address` column  
that could affect geocoding or display.

In [ ]:
print("=== ALL ADDRESSES ===")
for i, row in df.iterrows():
    print(f"  [{i:>2}] {row['Name']:<35} | {row['Address']}")

# Flag addresses missing a comma before state abbreviation (e.g., 'Malibu CA' vs 'Malibu, CA')
print("\n=== ADDRESSES MISSING COMMA BEFORE STATE ===")
import re
bad_format = df[df['Address'].str.contains(r'[a-zA-Z]\s+CA\s', regex=True) 
                & ~df['Address'].str.contains(r',\s*CA', regex=True)]
if bad_format.empty:
    print("✅ All addresses have proper comma before state.")
else:
    print(f"⚠️  {len(bad_format)} address(es) missing comma before CA:")
    print(bad_format[['Name', 'Address']].to_string())

## 🛠️ Cell 12 — Fix Known Address Issues

**Issues identified during inspection:**
1. `Mastro's Ocean Club` — missing comma before `CA` in address  
2. `Morihiro` — listed in Echo Park but address has zip code `90012` (Downtown). Correct Echo Park zip is `90026`  
3. `Restaurant Ki`, `Rasarumah`, `H&H Brazilian` — trailing spaces in address (already fixed in Cell 9)

In [ ]:
# --- Fix 1: Mastro's Ocean Club — add missing comma before CA ---
mastros_idx = df[df['Name'] == "Mastro's Ocean Club"].index
if not mastros_idx.empty:
    old_addr = df.loc[mastros_idx[0], 'Address']
    fixed_addr = old_addr.replace('Malibu CA', 'Malibu, CA')
    df.loc[mastros_idx[0], 'Address'] = fixed_addr
    print(f"✅ Fixed Mastro's Ocean Club address:")
    print(f"   Before: {old_addr}")
    print(f"   After : {fixed_addr}")

# --- Fix 2: Morihiro — correct zip code from 90012 (DTLA) to 90026 (Echo Park) ---
morihiro_idx = df[df['Name'] == 'Morihiro'].index
if not morihiro_idx.empty:
    old_addr = df.loc[morihiro_idx[0], 'Address']
    # Correct zip: Sunset Blvd near Echo Park is 90026
    fixed_addr = old_addr.replace('CA 90012', 'CA 90026')
    df.loc[morihiro_idx[0], 'Address'] = fixed_addr
    print(f"\n✅ Fixed Morihiro address zip code:")
    print(f"   Before: {old_addr}")
    print(f"   After : {fixed_addr}")

print("\n✅ Address fixes applied.")

## 📝 Cell 13 — Description Quality Check

Verifies all descriptions are meaningful (not too short) and unique.

In [ ]:
# Word count per description
df['desc_word_count'] = df['Description'].str.split().str.len()

print("=== DESCRIPTION WORD COUNT STATS ===")
print(df['desc_word_count'].describe())

# Flag descriptions that are too short (< 10 words)
short_desc = df[df['desc_word_count'] < 10]
print(f"\n=== DESCRIPTIONS UNDER 10 WORDS ===")
if short_desc.empty:
    print("✅ All descriptions meet minimum word count (10+ words).")
else:
    print(f"⚠️  {len(short_desc)} description(s) are very short:")
    print(short_desc[['Name', 'Description', 'desc_word_count']].to_string())

# Check for duplicate descriptions
dupe_desc = df[df.duplicated('Description', keep=False)]
print(f"\n=== DUPLICATE DESCRIPTIONS ===")
if dupe_desc.empty:
    print("✅ All descriptions are unique.")
else:
    print(f"⚠️  {len(dupe_desc)} rows share identical descriptions:")
    print(dupe_desc[['Name', 'Description']].to_string())

## 📞 Cell 14 — Telephone Number Validation

In [ ]:
import re

print("=== TELEPHONE NUMBER FORMAT CHECK ===")

# Expected formats: (XXX) XXX-XXXX or 'Number Not Listed'
phone_pattern = re.compile(r'^\(\d{3}\) \d{3}-\d{4}$|^Number Not Listed$')

invalid_phones = df[~df['Telephone Number'].str.match(phone_pattern)]

if invalid_phones.empty:
    print("✅ All phone numbers are correctly formatted.")
else:
    print(f"⚠️  {len(invalid_phones)} phone number(s) with unexpected format:")
    print(invalid_phones[['Name', 'Telephone Number']].to_string())

# Summary of phone listing status
print(f"\n=== PHONE NUMBER STATUS ===")
listed = (df['Telephone Number'] != 'Number Not Listed').sum()
not_listed = (df['Telephone Number'] == 'Number Not Listed').sum()
print(f"  Listed       : {listed}")
print(f"  Not Listed   : {not_listed}")

## 💰 Cell 15 — Price Range Validation

In [ ]:
print("=== PRICE RANGE DISTRIBUTION ===")

valid_prices = ['$', '$$', '$$$', '$$$$', '$$$$$']
price_counts = df['Price'].value_counts()

# Display in logical order
for price in valid_prices:
    count = price_counts.get(price, 0)
    bar = '█' * count
    print(f"  {price:<8} | {bar} ({count})")

# Check for invalid values
invalid_prices = df[~df['Price'].isin(valid_prices)]
if invalid_prices.empty:
    print("\n✅ All price values are valid.")
else:
    print(f"\n⚠️  {len(invalid_prices)} invalid price value(s):")
    print(invalid_prices[['Name', 'Price']].to_string())

## ⭐ Cell 16 — Michelin Guide Validation

In [ ]:
print("=== MICHELIN GUIDE DISTRIBUTION ===")

valid_michelin = ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']
michelin_order = ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']
michelin_counts = df['Michelin-Guide'].value_counts()

for m in michelin_order:
    count = michelin_counts.get(m, 0)
    bar = '★' * count
    print(f"  {m:<20} | {bar} ({count})")

# Invalid check
invalid_michelin = df[~df['Michelin-Guide'].isin(valid_michelin)]
if invalid_michelin.empty:
    print("\n✅ All Michelin-Guide values are valid.")
else:
    print(f"\n⚠️  Unexpected Michelin values:")
    print(invalid_michelin[['Name', 'Michelin-Guide']].to_string())

# Show all Michelin starred restaurants
print("\n=== ALL MICHELIN-STARRED RESTAURANTS ===")
starred = df[df['Michelin-Guide'].isin(['3-Star', '2-Star', '1-Star'])].sort_values('Michelin-Guide')
print(starred[['Name', 'Location', 'Michelin-Guide', 'Price', 'Customer Ratings']].to_string(index=True))

## 🌆 Cell 17 — Sky-High Rooftop & Atmosphere Validation

In [ ]:
# Sky-High Rooftop check
print("=== SKY-HIGH ROOFTOP ===")
valid_rooftop = ['Yes', 'No']
print(df['Sky-High Rooftop'].value_counts())

invalid_rooftop = df[~df['Sky-High Rooftop'].isin(valid_rooftop)]
if invalid_rooftop.empty:
    print("✅ All Sky-High Rooftop values are valid.")
else:
    print("⚠️  Invalid values:", invalid_rooftop[['Name', 'Sky-High Rooftop']].to_string())

# Rooftop restaurants list
print("\n=== ROOFTOP / TOP-FLOOR RESTAURANTS ===")
rooftops = df[df['Sky-High Rooftop'] == 'Yes']
print(rooftops[['Name', 'Location', 'Dining Atmosphere', 'Price']].to_string(index=True))

# Dining Atmosphere check
print("\n=== DINING ATMOSPHERE DISTRIBUTION ===")
print(df['Dining Atmosphere'].value_counts())

valid_atmosphere = ['Fine-Dining', 'Trendy', 'Casual', 'Romantic', 
                    'Smart-Casual', 'Romantic / Smart-Casual', 'Trendy / Smart-Casual']
invalid_atm = df[~df['Dining Atmosphere'].isin(valid_atmosphere)]
if invalid_atm.empty:
    print("\n✅ All Dining Atmosphere values are valid.")
else:
    print("\n⚠️  Unexpected atmosphere values:")
    print(invalid_atm[['Name', 'Dining Atmosphere']].to_string())

## 🕐 Cell 18 — Reservations & Dress Code Validation

In [ ]:
# Reservations check
print("=== RESERVATIONS DISTRIBUTION ===")
valid_reservations = ['Yes', 'No', 'Reservation Only']
print(df['Reservations'].value_counts())

invalid_res = df[~df['Reservations'].isin(valid_reservations)]
if invalid_res.empty:
    print("✅ All Reservations values are valid.")
else:
    print("⚠️  Unexpected values:", invalid_res[['Name', 'Reservations']].to_string())

# Dress Code check
print("\n=== DRESS CODE DISTRIBUTION ===")
valid_dresscode = ['Yes', 'No', 'Strict']
print(df['Dress Code'].value_counts())

invalid_dress = df[~df['Dress Code'].isin(valid_dresscode)]
if invalid_dress.empty:
    print("✅ All Dress Code values are valid.")
else:
    print("⚠️  Unexpected values:", invalid_dress[['Name', 'Dress Code']].to_string())

# Reservation-Only restaurants (most exclusive)
print("\n=== RESERVATION-ONLY RESTAURANTS ===")
res_only = df[df['Reservations'] == 'Reservation Only']
print(res_only[['Name', 'Location', 'Michelin-Guide', 'Price']].to_string(index=True))

## ⭐ Cell 19 — Customer Ratings Validation

In [ ]:
print("=== CUSTOMER RATINGS VALIDATION ===")

# Valid range: 1.0 to 5.0
out_of_range = df[(df['Customer Ratings'] < 1.0) | (df['Customer Ratings'] > 5.0)]
if out_of_range.empty:
    print("✅ All Customer Ratings are within valid range (1.0–5.0).")
else:
    print(f"⚠️  {len(out_of_range)} rating(s) out of range:")
    print(out_of_range[['Name', 'Customer Ratings']].to_string())

print("\n=== RATING SUMMARY ===")
print(f"  Average Rating : {df['Customer Ratings'].mean():.2f}")
print(f"  Highest Rating : {df['Customer Ratings'].max()} ({df[df['Customer Ratings'] == df['Customer Ratings'].max()]['Name'].tolist()})")
print(f"  Lowest Rating  : {df['Customer Ratings'].min()} ({df[df['Customer Ratings'] == df['Customer Ratings'].min()]['Name'].tolist()})")

print("\n=== TOP 10 HIGHEST RATED RESTAURANTS ===")
top10 = df.nlargest(10, 'Customer Ratings')[['Name', 'Location', 'Michelin-Guide', 'Price', 'Customer Ratings']]
print(top10.to_string(index=True))

## 🤖 Cell 20 — Create `restaurant_metadata` Column for LLM/Embedding Use

This column combines key restaurant metadata into a single enriched text string.  
It will be used as input to the sentence embedding model (`all-MiniLM-L6-v2`)  
for semantic search and LLM-based recommendation retrieval.

In [ ]:
def build_restaurant_metadata(row):
    """
    Combines restaurant metadata into a rich text string for embedding.
    Format: '[Name] is a [Cuisine Type] restaurant in [Location], [CA].
             [Description] Price: [Price]. Atmosphere: [Atmosphere].
             Michelin: [Michelin]. Rating: [Rating]/5."
    """
    michelin_text = (
        f"Michelin {row['Michelin-Guide']}" 
        if row['Michelin-Guide'] not in ['No', 'Michelin-Selected'] 
        else row['Michelin-Guide']
    )
    rooftop_text = " Rooftop/top-floor dining." if row['Sky-High Rooftop'] == 'Yes' else ""
    
    return (
        f"{row['Name']} is a {row['Cuisine Type']} restaurant located in "
        f"{row['Location']}, Los Angeles. {row['Description']} "
        f"Price range: {row['Price']}. Atmosphere: {row['Dining Atmosphere']}. "
        f"Michelin Guide: {michelin_text}. Customer Rating: {row['Customer Ratings']}/5."
        f"{rooftop_text}"
    )

df['restaurant_metadata'] = df.apply(build_restaurant_metadata, axis=1)

print("✅ 'restaurant_metadata' column created.")
print("\n=== SAMPLE RESTAURANT METADATA ===")
for i in [0, 6, 13, 26, 36]:  # Sample: Somni, Sushi Kaneyoshi, Holbox, Maccheroni, 71Above
    print(f"\n[{df.loc[i, 'Name']}]")
    print(f"  {df.loc[i, 'restaurant_metadata']}")

## 📋 Cell 21 — Full Dataset Review (All 100 Restaurants)

In [ ]:
print("=== COMPLETE CLEANED DATASET ===")
display_cols = [
    'Name', 'Location', 'Price', 'Cuisine Type', 
    'Dining Atmosphere', 'Michelin-Guide', 'Customer Ratings', 
    'Sky-High Rooftop', 'Reservations', 'Dress Code'
]
df[display_cols]

## 📊 Cell 22 — Final Data Summary Report

In [ ]:
print("="*60)
print("   FINAL DATASET SUMMARY REPORT")
print("="*60)
print(f"  Total Restaurants        : {len(df)}")
print(f"  Total Columns            : {len(df.columns)}")
print(f"  Missing Values           : {df.drop(columns=['desc_word_count']).isnull().sum().sum()}")
print(f"  Exact Duplicate Rows     : {df.duplicated(subset=['Name','Address']).sum()}")
print()
print("  --- Michelin Breakdown ---")
for m in ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']:
    count = (df['Michelin-Guide'] == m).sum()
    print(f"  {m:<22}: {count}")
print()
print("  --- Price Breakdown ---")
for p in ['$$$$$', '$$$$', '$$$', '$$', '$']:
    count = (df['Price'] == p).sum()
    print(f"  {p:<22}: {count}")
print()
print(f"  Average Customer Rating  : {df['Customer Ratings'].mean():.2f} / 5.0")
print(f"  Rooftop Restaurants      : {(df['Sky-High Rooftop'] == 'Yes').sum()}")
print(f"  Reservation-Only Spots   : {(df['Reservations'] == 'Reservation Only').sum()}")
print(f"  Strict Dress Code        : {(df['Dress Code'] == 'Strict').sum()}")
print()
print(f"  restaurant_metadata      : {df['restaurant_metadata'].notna().sum()} / {len(df)} created")
print("="*60)
print("  ✅ Dataset is clean and ready for vector embedding!")
print("="*60)

## 💾 Cell 23 — Export Cleaned Dataset

Saves the final cleaned and enriched dataset.  
The temporary helper column `desc_word_count` is dropped before export.

In [ ]:

df_final = df.drop(columns=['desc_word_count'])


output_path = "../data/cleaned_restaurants_final.csv"
df_final.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset exported to: {output_path}")
print(f"   Rows    : {df_final.shape[0]}")
print(f"   Columns : {df_final.shape[1]}")
print(f"\nFinal columns:")
for i, col in enumerate(df_final.columns, 1):
    print(f"   {i:>2}. {col}")

print("\n🚀 Next step: Use cleaned_restaurants_final.csv in notebook 2 — Vector Embedding & Semantic Search")